# Coupling calibration outputs

This notebook directly regenerates the paper figures and numerical summaries for the coupling section.

The calculations use the calibrated marginal CARMA outputs and the coupling artefacts. The raw residual correlations are diagnostic only; the coupling estimator uses filtered state-space residuals and recovered driver increments.

In [1]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import pearsonr

ROOT = Path.cwd().parents[1]
PRICE_SEAS = ROOT / "germany" / "germany23+24+25" / "data" / "seasonality"
OUT = ROOT / "newpaper" / "figures" / "coupling"
OUT.mkdir(parents=True, exist_ok=True)

BLUE = "#1f4e79"
BLACK = "#111111"
GRID = "#d9d9d9"

plt.rcParams.update(
    {
        "font.size": 9,
        "axes.grid": True,
        "grid.color": GRID,
        "grid.alpha": 0.55,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "legend.frameon": False,
    }
)


def fmt_sig(x, digits=2):
    if x == 0:
        return "0.0"
    decimals = max(digits - int(np.floor(np.log10(abs(x)))) - 1, 0)
    return f"{x:.{decimals}f}"


def savefig(fig, name):
    fig.savefig(OUT / f"{name}.pdf", bbox_inches="tight")
    fig.savefig(OUT / f"{name}.png", dpi=180, bbox_inches="tight")


def berlin_naive_to_utc(index):
    timestamps = pd.Series(pd.to_datetime(index))
    localized = timestamps.dt.tz_localize("Europe/Berlin", ambiguous="NaT", nonexistent="NaT")
    return pd.DatetimeIndex(localized.dt.tz_convert("UTC"))


def read_price_residuals():
    price_panel = pd.read_csv(PRICE_SEAS / "german_panel.csv", parse_dates=["datetime"])
    return pd.DataFrame(
        {
            "datetime": pd.to_datetime(price_panel["datetime"], utc=True),
            "price_residual": price_panel["log_price_resid"].to_numpy(float),
        }
    ).dropna()


def compute_coupling_outputs(cfg):
    coupling_dir = cfg["coupling_dir"]
    carma_dir = cfg["carma_dir"]

    with open(coupling_dir / cfg["result_json"], "r", encoding="utf-8") as fh:
        result = json.load(fh)

    fit_table = pd.read_csv(coupling_dir / "price_idiosyncratic_driver_fit_comparison.csv")
    diagnostics = pd.read_csv(coupling_dir / cfg["diagnostics_csv"]).iloc[0]
    aligned = np.load(coupling_dir / cfg["aligned_npz"], allow_pickle=True)

    d_factor = aligned[cfg["factor_driver_key"]].astype(float)
    d_price_marginal = aligned["dW_price_marginal"].astype(float)
    d_price_idio = aligned["dL_price_idio"].astype(float)
    lambda_component = aligned[cfg["lambda_component_key"]].astype(float)

    gaussian_row = fit_table.loc[fit_table["model"].eq("Gaussian")].iloc[0]
    nig_row = fit_table.loc[fit_table["model"].eq("NIG")].iloc[0]

    factor_panel = pd.read_csv(carma_dir / cfg["latent_panel"], index_col=0, parse_dates=True)
    if cfg.get("berlin_index", False):
        factor_datetime = berlin_naive_to_utc(factor_panel.index)
    else:
        factor_datetime = pd.to_datetime(factor_panel.index, utc=True)

    factor_df = pd.DataFrame(
        {
            "datetime": factor_datetime,
            "factor_residual": factor_panel[cfg["latent_col"]].to_numpy(float),
        }
    ).dropna()

    raw = (
        factor_df.drop_duplicates("datetime")
        .merge(read_price_residuals().drop_duplicates("datetime"), on="datetime", how="inner")
        .sort_values("datetime")
    )
    raw_corr_test = pearsonr(raw["factor_residual"], raw["price_residual"])

    fig, ax = plt.subplots(figsize=(6.2, 4.4))
    ax.scatter(
        raw["factor_residual"],
        raw["price_residual"],
        s=7,
        alpha=0.20,
        color=BLUE,
        edgecolors="none",
        rasterized=True,
    )
    ax.axhline(0.0, color=BLACK, lw=0.7, alpha=0.45)
    ax.axvline(0.0, color=BLACK, lw=0.7, alpha=0.45)
    ax.set_title(f"Raw residual levels, correlation = {fmt_sig(raw_corr_test.statistic)}")
    ax.set_xlabel(cfg["x_label"])
    ax.set_ylabel(r"log-price residual $Y^P_t$")
    fig.tight_layout()
    savefig(fig, f"{cfg['prefix']}_price_raw_residual_scatter")
    plt.close(fig)

    corr_before_test = pearsonr(d_price_marginal, d_factor)
    corr_after_test = pearsonr(d_price_idio, d_factor)
    std_lambda = float(np.std(lambda_component, ddof=1))
    std_idio = float(np.std(d_price_idio, ddof=1))
    std_gaussian_idio = float(gaussian_row["std"])

    summary = {
        "raw_residual_corr": float(raw_corr_test.statistic),
        "lambda_hat": float(result["lambda_hat"]),
        "state_residual_output_corr": float(diagnostics["state_residual_output_corr"]),
        "corr_before": float(corr_before_test.statistic),
        "corr_after": float(corr_after_test.statistic),
        "std_idio": std_idio,
        "std_gaussian_idio": std_gaussian_idio,
        "std_nig_idio": float(nig_row["std"]),
        f"std_lambda_{cfg['prefix']}": std_lambda,
        "std_lambda_to_idio_ratio": std_lambda / std_idio,
        f"mean_lambda_{cfg['prefix']}": float(np.mean(lambda_component)),
        "n_common": int(result["n_common_hourly_intervals"]),
    }
    pd.Series(summary).to_csv(OUT / f"{cfg['prefix']}_price_coupling_summary.csv", header=["value"])
    return pd.Series(summary)


## Temperature--price coupling

In [2]:
temperature_cfg = {
    "prefix": "temperature",
    "coupling_dir": ROOT / "temperature" / "data" / "coupling",
    "carma_dir": ROOT / "temperature" / "data" / "carma",
    "result_json": "temperature_price_coupling_result.json",
    "diagnostics_csv": "temperature_price_lambda_diagnostics.csv",
    "aligned_npz": "temperature_price_aligned_drivers.npz",
    "factor_driver_key": "dW_temperature",
    "lambda_component_key": "lambda_temperature_component",
    "latent_panel": "temperature_latent_panel.csv",
    "latent_col": "temperature_XtQ",
    "x_label": r"temperature residual $Y^T_t$",
}

temperature_summary = compute_coupling_outputs(temperature_cfg)
temperature_summary


raw_residual_corr            -2.387081e-01
lambda_hat                   -1.368883e-03
state_residual_output_corr   -4.049980e-02
corr_before                  -4.159236e-02
corr_after                   -9.388156e-04
std_idio                      1.322465e-02
std_gaussian_idio             1.322440e-02
std_nig_idio                  1.212735e-02
std_lambda_temperature        5.381052e-04
std_lambda_to_idio_ratio      4.068955e-02
mean_lambda_temperature       3.536045e-07
n_common                      2.630300e+04
dtype: float64

## Solar--price coupling

In [3]:
solar_cfg = {
    "prefix": "solar",
    "coupling_dir": ROOT / "solar" / "Intensity_Model_solar" / "data" / "coupling",
    "carma_dir": ROOT / "solar" / "Intensity_Model_solar" / "data" / "carma",
    "result_json": "solar_price_coupling_result.json",
    "diagnostics_csv": "solar_price_lambda_diagnostics.csv",
    "aligned_npz": "solar_price_aligned_drivers.npz",
    "factor_driver_key": "dW_solar",
    "lambda_component_key": "lambda_solar_component",
    "latent_panel": "solar_latent_panel.csv",
    "latent_col": "solar_XtQ",
    "x_label": r"solar latent residual $Y^S_t$",
    "berlin_index": True,
}

solar_summary = compute_coupling_outputs(solar_cfg)
solar_summary


raw_residual_corr             6.267017e-02
lambda_hat                   -7.968459e-06
state_residual_output_corr   -1.795055e-03
corr_before                  -1.001688e-03
corr_after                    7.781115e-04
std_idio                      1.272117e-02
std_gaussian_idio             1.272089e-02
std_nig_idio                  1.161281e-02
std_lambda_solar              2.264113e-05
std_lambda_to_idio_ratio      1.779800e-03
mean_lambda_solar             5.480450e-08
n_common                      2.310300e+04
dtype: float64

## Wind--price coupling

In [4]:
wind_cfg = {
    "prefix": "wind",
    "coupling_dir": ROOT / "wind" / "carma_coupling" / "data" / "coupling",
    "carma_dir": ROOT / "wind" / "carma_coupling" / "data" / "carma",
    "result_json": "wind_price_coupling_result.json",
    "diagnostics_csv": "wind_price_lambda_diagnostics.csv",
    "aligned_npz": "wind_price_aligned_drivers.npz",
    "factor_driver_key": "dW_wind",
    "lambda_component_key": "lambda_wind_component",
    "latent_panel": "wind_latent_panel.csv",
    "latent_col": "wind_XtQ",
    "x_label": r"wind logit residual $Y^W_t$",
}

wind_summary = compute_coupling_outputs(wind_cfg)
wind_summary


raw_residual_corr            -4.975757e-01
lambda_hat                   -1.691296e-03
state_residual_output_corr   -4.024098e-02
corr_before                  -4.023664e-02
corr_after                    3.685755e-05
std_idio                      1.322517e-02
std_gaussian_idio             1.322491e-02
std_nig_idio                  1.211537e-02
std_lambda_wind               5.330549e-04
std_lambda_to_idio_ratio      4.030611e-02
mean_lambda_wind              4.708512e-07
n_common                      2.630400e+04
dtype: float64